In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris


In [3]:
data = load_iris()

df = pd.DataFrame(data.data, columns=data.feature_names)

df['target'] = (data.target != 0).astype(int)

print(df.head())

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  


In [4]:
X = df.drop('target', axis=1).to_numpy()
y = df['target'].to_numpy()

In [5]:
y = y.reshape(-1, 1)
y.shape

(150, 1)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, train_size=0.8)

In [7]:
X_train = (X_train - X_train.mean(axis=0)) / X_train.std(axis=0)
X_test = (X_test - X_test.mean(axis=0)) / X_test.std(axis=0)

In [8]:
w_h = np.random.rand(X.shape[1], 6)
b_h = np.random.rand(6)

w_o = np.random.rand(6, 1)
b_o = np.random.rand(1)

In [9]:
w_h

array([[0.74456509, 0.63002639, 0.47777906, 0.08870123, 0.5511035 ,
        0.52082815],
       [0.83356863, 0.28590968, 0.64197872, 0.00101686, 0.01915718,
        0.96851014],
       [0.63452668, 0.47511929, 0.82412875, 0.33262529, 0.49377641,
        0.14954238],
       [0.48668382, 0.80744968, 0.40692988, 0.35208846, 0.25383144,
        0.81194427]])

In [14]:
def sigmoid(x):
    return 1 / (1+np.exp(-x))

def sig_der(x):
    return x * (1-x)

def relu(x):
    return np.maximum(0, x)

def relu_der(x):
    return (x > 0).astype(int)

def tanh(x):
    return np.tanh(x)

def tanh_der(x):
    return 1 - x**2

In [ ]:
act_hidden = "tanh"        # try: sigmoid / tanh / relu
act_output = "sigmoid"     # sigmoid for binary
loss_type = "bce"          # mse / bce

epochs = 1000

for _ in range(epochs):
    # forward
    z_hidden = X_train @ w_h + b_h
    a_hidden = activation(z_hidden, act_hidden)

    z_output = a_hidden @ w_o + b_o

    if loss_type == "cce":
        y_pred = softmax(z_output)
    else:
        y_pred = activation(z_output, act_output)

    # loss derivative
    if loss_type == "mse":
        del_output = (y_pred - y_train) * activation_der(y_pred, act_output)
    elif loss_type == "bce":
        del_output = (y_pred - y_train)
    elif loss_type == "cce":
        del_output = (y_pred - y_train)

    # hidden delta
    if act_hidden == "relu":
        del_hidden = (del_output @ w_o.T) * activation_der(z_hidden, act_hidden)
    else:
        del_hidden = (del_output @ w_o.T) * activation_der(a_hidden, act_hidden)

    # update
    w_o -= lr * (a_hidden.T @ del_output)
    b_o -= lr * del_output.sum(axis=0)

    w_h -= lr * (X_train.T @ del_hidden)
    b_h -= lr * del_hidden.sum(axis=0)

In [12]:
y_pred = []

for x in X_test:
    z_hidden = x @ w_h + b_h
    z = sigmoid(z_hidden)

    z_output = z @ w_o + b_o
    y_hat = sigmoid(z_output)

    y_pred.append(1 if y_hat >= 0.5 else 0)

In [13]:
from sklearn.metrics import classification_report, accuracy_score

print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        10
           1       0.67      1.00      0.80        20

    accuracy                           0.67        30
   macro avg       0.33      0.50      0.40        30
weighted avg       0.44      0.67      0.53        30

0.6666666666666666


/Users/aryanahuja/Documents/Aryan/Projects/DL_Prac/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/aryanahuja/Documents/Aryan/Projects/DL_Prac/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/aryanahuja/Documents/Aryan/Projects/DL_Prac/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contr